# 02 — Graph Construction

**Goal:** Turn the raw ingested data into a `GraphFrame` — the core data structure for all analytics in notebook 03.

## What this notebook produces
| Output file | Contents |
|---|---|
| `processed/graph_vertices.parquet` | Node table (packages + contributors) |
| `processed/graph_edges.parquet` | Edge table (DEPENDS_ON + MAINTAINS) |
| `sample/graph_vertices_10k.parquet` | Same, 10k-package sample |
| `sample/graph_edges_10k.parquet` | Same, 10k-package sample |

## Learning note: Why a directed graph?
Dependency graphs are **directed** — if package A depends on B, the risk flows from B *to* A (a vulnerability in B impacts A, not the other way around). In GraphFrames terms, the edge `(B) -[DEPENDS_ON]-> (A)` means "B is upstream of A". PageRank will naturally flow weight toward the nodes with the most inbound dependency edges — i.e., the most-depended-upon packages — which is exactly what we want to measure.

## Learning note: GraphFrames vs NetworkX
For small graphs (<100k nodes), you could use `networkx`. We use **GraphFrames** (PySpark) because:
1. It scales to millions of nodes without memory issues
2. Its PageRank and Label Propagation implementations are parallelized
3. Motif finding (pattern matching on graph structure) is far cleaner than NetworkX

The downside: PySpark has a non-trivial setup in Colab (covered in Section 0).

## 0 — Setup: PySpark + GraphFrames in Colab

**Why the extra steps?** GraphFrames requires a specific JAR file that isn't bundled with PySpark. We have to tell Spark where to find it. The `--packages` flag does this automatically by pulling the JAR from Maven Central at Spark session start.

In [ ]:
!pip install -q pyspark==3.5.0 graphframes pandas pyarrow

In [ ]:
import os

# Tell PySpark to download the GraphFrames JAR automatically
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages graphframes:graphframes:0.8.3-spark3.5-s_2.12 pyspark-shell'
)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, lower, when, datediff, current_date

spark = (
    SparkSession.builder
    .appName('BlastRadius-GraphConstruction')
    # Give Spark as much RAM as Colab Pro allows
    .config('spark.driver.memory', '20g')
    .config('spark.sql.shuffle.partitions', '50')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/BlastRadius'

# Toggle USE_SAMPLE = True during development, False for the full-scale run
USE_SAMPLE = True
DATA_DIR   = f'{BASE}/data/sample' if USE_SAMPLE else f'{BASE}/data/processed'
OUT_DIR    = f'{BASE}/data/sample' if USE_SAMPLE else f'{BASE}/data/processed'

print(f'USE_SAMPLE = {USE_SAMPLE}')
print(f'Reading from: {DATA_DIR}')

## 1 — Load ingested data

In [ ]:
import pandas as pd

suffix = '_10k' if USE_SAMPLE else ''

packages_pd = pd.read_parquet(f'{DATA_DIR}/npm_packages{suffix}.parquet')
deps_pd     = pd.read_parquet(f'{DATA_DIR}/npm_dependencies{suffix}.parquet')
commits_pd  = pd.read_parquet(f'{DATA_DIR}/gh_commits{suffix}.parquet')

print(f'Packages:     {len(packages_pd):,}')
print(f'Dependencies: {len(deps_pd):,}')
print(f'Commit rows:  {len(commits_pd):,}')

In [ ]:
# Convert to Spark DataFrames
packages_sdf = spark.createDataFrame(packages_pd)
deps_sdf     = spark.createDataFrame(deps_pd)
commits_sdf  = spark.createDataFrame(commits_pd)

packages_sdf.printSchema()

## 2 — Build the vertex table

**Why two node types?** Our graph has two kinds of nodes:
1. **Package nodes** — represent an NPM package (e.g., `react`, `lodash`)
2. **Contributor nodes** — represent a GitHub user (e.g., `gaearon`, `jdalton`)

GraphFrames requires a single `vertices` DataFrame with a column named `id` (must be unique and a string). We prefix IDs to avoid collisions: `pkg:react`, `user:gaearon`.

The `node_type` column lets us filter to only package nodes later when we compute PageRank, since we only want to rank packages — not contributors.

In [ ]:
from pyspark.sql.functions import concat, regexp_replace

# Package nodes
package_nodes = (
    packages_sdf
    .select(
        concat(lit('pkg:'), lower(col('name'))).alias('id'),
        lower(col('name')).alias('name'),
        lit('package').alias('node_type'),
        col('dependent_packages'),
        col('dependent_repos'),
        col('stars'),
        col('last_publish'),
        col('github_slug'),
        col('status')
    )
)

print(f'Package nodes: {package_nodes.count():,}')
package_nodes.show(3, truncate=False)

In [ ]:
from pyspark.sql.functions import countDistinct

# Contributor nodes — one node per unique GitHub login
# We derive these from the commit data (not from npm — we want people who actually commit)
contributor_nodes = (
    commits_sdf
    .select(col('author_login'))
    .distinct()
    .filter(col('author_login').isNotNull())
    .select(
        concat(lit('user:'), lower(col('author_login'))).alias('id'),
        lower(col('author_login')).alias('name'),
        lit('contributor').alias('node_type'),
        lit(None).cast('long').alias('dependent_packages'),
        lit(None).cast('long').alias('dependent_repos'),
        lit(None).cast('long').alias('stars'),
        lit(None).cast('timestamp').alias('last_publish'),
        lit(None).cast('string').alias('github_slug'),
        lit(None).cast('string').alias('status')
    )
)

print(f'Contributor nodes: {contributor_nodes.count():,}')
contributor_nodes.show(3, truncate=False)

In [ ]:
# Union into a single vertices DataFrame
vertices = package_nodes.unionByName(contributor_nodes)

total_nodes = vertices.count()
print(f'Total vertices: {total_nodes:,}')
vertices.groupBy('node_type').count().show()

## 3 — Build the edge table

**Why two edge types?** We model two fundamentally different relationships:

1. **`DEPENDS_ON`** (`Package → Package`) — structural dependency. This is the blast-radius edge: if B has a vulnerability, all packages that `DEPENDS_ON` B (directly or transitively) are affected.

2. **`MAINTAINS`** (`Contributor → Package`) — human maintenance relationship. This is the Bus Factor edge: if only one contributor `MAINTAINS` a package, that contributor's departure kills the project.

GraphFrames requires `src` and `dst` columns in the edge table (both must be valid vertex `id` values), plus an `edge_type` column for filtering.

In [ ]:
# DEPENDS_ON edges: Package → Package
# src = the package that has the dependency (downstream)
# dst = the package being depended on (upstream / the risk source)
dependency_edges = (
    deps_sdf
    .select(
        concat(lit('pkg:'), lower(col('src_package'))).alias('src'),
        concat(lit('pkg:'), lower(col('dst_package'))).alias('dst'),
        lit('DEPENDS_ON').alias('edge_type'),
        col('dep_kind'),
        col('is_optional')
    )
    # Only keep edges where both src and dst exist in our vertex table
    .join(
        vertices.select(col('id').alias('src_check')),
        col('src') == col('src_check')
    ).drop('src_check')
    .join(
        vertices.select(col('id').alias('dst_check')),
        col('dst') == col('dst_check')
    ).drop('dst_check')
)

print(f'DEPENDS_ON edges: {dependency_edges.count():,}')
dependency_edges.show(3, truncate=False)

In [ ]:
from pyspark.sql.functions import sum as spark_sum, max as spark_max, to_date

# MAINTAINS edges: Contributor → Package
# We need to join commits (which have github_slug) to packages (which have github_slug)
# to get the package name → contributor relationship

# Step 1: Aggregate commit data per (repo, author)
commit_agg = (
    commits_sdf
    .groupBy('github_slug', 'author_login')
    .agg(
        spark_sum('push_count').alias('total_commits'),
        spark_max('commit_date').alias('last_commit_date')
    )
)

# Step 2: Join with package nodes to get package IDs
package_slugs = (
    packages_sdf
    .select(lower(col('name')).alias('package_name'), col('github_slug'))
    .filter(col('github_slug').isNotNull())
)

maintains_edges = (
    commit_agg
    .join(package_slugs, 'github_slug')
    .select(
        concat(lit('user:'), lower(col('author_login'))).alias('src'),
        concat(lit('pkg:'), col('package_name')).alias('dst'),
        lit('MAINTAINS').alias('edge_type'),
        col('total_commits'),
        col('last_commit_date'),
        # Days since last commit — used for recency weighting
        datediff(current_date(), col('last_commit_date')).alias('days_since_commit'),
        lit(None).cast('string').alias('dep_kind'),
        lit(False).alias('is_optional')
    )
)

print(f'MAINTAINS edges: {maintains_edges.count():,}')
maintains_edges.show(3, truncate=False)

In [ ]:
# Union all edges into a single edge table
# We need both edge types to have the same columns for unionByName
dep_edges_aligned = dependency_edges.withColumn('total_commits', lit(None).cast('long')) \
                                     .withColumn('last_commit_date', lit(None).cast('date')) \
                                     .withColumn('days_since_commit', lit(None).cast('int'))

edges = dep_edges_aligned.unionByName(maintains_edges)

total_edges = edges.count()
print(f'Total edges: {total_edges:,}')
edges.groupBy('edge_type').count().show()

## 4 — Construct the GraphFrame

**Why:** GraphFrames wraps the vertex and edge DataFrames into a graph object that exposes PageRank, BFS, motif finding, and other algorithms. The `GraphFrame(v, e)` constructor just needs:
- `v`: a DataFrame with column `id` (string, unique)
- `e`: a DataFrame with columns `src` and `dst` (must be valid `id` values)

Everything else (any extra columns) is carried through as metadata.

In [ ]:
from graphframes import GraphFrame

g = GraphFrame(vertices, edges)

print('GraphFrame constructed:')
print(f'  Vertices: {g.vertices.count():,}')
print(f'  Edges:    {g.edges.count():,}')
print()
print('Vertex schema:')
g.vertices.printSchema()
print('Edge schema:')
g.edges.printSchema()

## 5 — Motif finding: diamond dependencies

**Why this is interesting:** A "diamond dependency" is when:
- Package A depends on B and D
- Both B and D depend on C

This means A has two *separate* paths to C. If C has a vulnerability, A is doubly exposed and the vulnerability may be hard to patch (different version requirements from B and D might conflict).

GraphFrames' motif syntax is a mini pattern-matching language: `(a)-[e1]->(b); (a)-[e2]->(d); (b)-[e3]->(c); (d)-[e4]->(c)` finds all subgraphs matching this shape.

In [ ]:
# Find diamond dependency patterns
# a → b → c AND a → d → c (two paths from a to c through different intermediaries)

# Filter to only DEPENDS_ON edges for this query
g_deps_only = GraphFrame(
    g.vertices.filter(col('node_type') == 'package'),
    g.edges.filter(col('edge_type') == 'DEPENDS_ON')
)

diamonds = g_deps_only.find(
    '(a)-[e1]->(b); (a)-[e2]->(d); (b)-[e3]->(c); (d)-[e4]->(c)'
).filter(
    # Ensure b and d are actually different packages
    col('b.id') != col('d.id')
).select(
    col('a.name').alias('package'),
    col('b.name').alias('path1_via'),
    col('d.name').alias('path2_via'),
    col('c.name').alias('shared_dep')
).distinct()

diamond_count = diamonds.count()
print(f'Diamond dependency patterns found: {diamond_count:,}')
if diamond_count > 0:
    diamonds.show(10, truncate=False)

## 6 — Persist graph to Drive

We save the vertex and edge tables to parquet so notebook 03 can reload them without re-running Spark. Parquet is a columnar binary format — much faster to read and much smaller than CSV.

In [ ]:
suffix = '_10k' if USE_SAMPLE else ''

vertices_path = f'{OUT_DIR}/graph_vertices{suffix}.parquet'
edges_path    = f'{OUT_DIR}/graph_edges{suffix}.parquet'

# Colab Drive writes work best with pandas for single-file parquet
g.vertices.toPandas().to_parquet(vertices_path, index=False)
g.edges.toPandas().to_parquet(edges_path, index=False)

import os
print(f'Vertices saved: {vertices_path}  ({os.path.getsize(vertices_path)/1e6:.1f} MB)')
print(f'Edges saved:    {edges_path}  ({os.path.getsize(edges_path)/1e6:.1f} MB)')

## 7 — Sanity checks

In [ ]:
# Check 1: Node/edge counts are reasonable
n_packages     = g.vertices.filter(col('node_type') == 'package').count()
n_contributors = g.vertices.filter(col('node_type') == 'contributor').count()
n_depends_on   = g.edges.filter(col('edge_type') == 'DEPENDS_ON').count()
n_maintains    = g.edges.filter(col('edge_type') == 'MAINTAINS').count()

print('CHECK 1 — graph shape:')
print(f'  Package nodes:      {n_packages:,}')
print(f'  Contributor nodes:  {n_contributors:,}')
print(f'  DEPENDS_ON edges:   {n_depends_on:,}  (expect >> packages)')
print(f'  MAINTAINS edges:    {n_maintains:,}')

avg_deps = n_depends_on / max(n_packages, 1)
print(f'  Avg dependencies per package: {avg_deps:.1f}')

In [ ]:
# Check 2: Most-depended-upon packages (inbound DEPENDS_ON edges)
# These should be recognisable names like lodash, chalk, semver, etc.
print('CHECK 2 — top 10 packages by inbound dependency count (most depended-upon):')
(
    g.edges
    .filter(col('edge_type') == 'DEPENDS_ON')
    .groupBy('dst')
    .count()
    .join(g.vertices.select(col('id'), col('name')), col('dst') == col('id'))
    .select('name', col('count').alias('inbound_deps'))
    .orderBy(col('inbound_deps').desc())
    .show(10, truncate=False)
)

In [ ]:
# Check 3: Packages with the most contributors (high bus factor candidates)
print('CHECK 3 — top 10 packages by unique contributor count:')
(
    g.edges
    .filter(col('edge_type') == 'MAINTAINS')
    .groupBy('dst')
    .count()
    .join(g.vertices.select(col('id'), col('name')), col('dst') == col('id'))
    .select('name', col('count').alias('contributor_count'))
    .orderBy(col('contributor_count').desc())
    .show(10, truncate=False)
)

In [ ]:
# Check 4: react's dependencies (spot-check graph correctness)
print('CHECK 4 — direct dependencies of react:')
(
    g.edges
    .filter((col('edge_type') == 'DEPENDS_ON') & (col('src') == 'pkg:react'))
    .join(g.vertices.select(col('id'), col('name')), col('dst') == col('id'))
    .select('name', 'dep_kind')
    .show(20, truncate=False)
)

In [ ]:
print('Graph construction complete.')
print('Next: open 03_analytics.ipynb')